# Lecture 05 — Classification Trees

**Term:** Fall 2025  
**Week/Topic:** Lecture 05
**Instructor:** Dr. Bushaj

---

### What you’ll learn
- When to use decision trees and how they work (splits, impurity, depth).
- Training a baseline `DecisionTreeClassifier`.
- Preventing overfitting via **max depth**, **min samples**, **cost-complexity pruning**.
- Evaluating models with **train/validation split** and **cross‑validation**.
- Interpreting trees (feature importance, text export, simple plots).
- (Optional) Hyperparameter tuning with `GridSearchCV` / `RandomizedSearchCV`.
- Clear, reproducible workflow with random seeds and stratification.

## Prerequisites
- Comfortable with **NumPy**, **Pandas**, **Matplotlib/Seaborn**.
- Basic classification metrics: accuracy, precision, recall, F1, ROC‑AUC.

## Dataset & Libraries
- Any tabular classification dataset with a labeled target column.
- Core libraries: `pandas`, `numpy`, `matplotlib`, `seaborn`, `sklearn`.

> Tip: If your classes are imbalanced, prefer **stratified splits**, monitor **precision/recall**, and consider **class weights**.

## Import Libraries

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
import matplotlib.pylab as plt
%pip install dmba
from dmba import plotDecisionTree, classificationSummary, regressionSummary
from sklearn import tree
import seaborn as sns

## Read Data

In [ ]:
my_drive_path = "YOUR_FILE_PATH_HERE"

In [ ]:
mower_df = pd.read_csv(my_drive_path + "RidingMowers.csv")

X = mower_df.drop(columns="Ownership")
y = mower_df["Ownership"]

## Create a Classifier

In [ ]:
# Creating a DecisionTreeClassifier
clf = DecisionTreeClassifier(random_state=0, max_depth=1)

# Fit the model
clf.fit(X, y)

In [ ]:
help(DecisionTreeClassifier)

In [ ]:
help(clf.fit)

In [ ]:
# Print the classes (target variable values) in the dataset
print("Classes: {}".format(', '.join(clf.classes_)))
# Visualize the decision tree classifier using the plotDecisionTree function
# Specify feature names for the plot and class names for the classes in the dataset
plotDecisionTree(clf, feature_names=mower_df.columns[:2], class_names=clf.classes_)

In [ ]:
help(plotDecisionTree)

## Grow the full tree

In [ ]:
classTree = DecisionTreeClassifier(random_state=0)
classTree.fit(X, y)

print("Classes: {}".format(', '.join(classTree.classes_)))

plotDecisionTree(classTree, feature_names=mower_df.columns[:2], class_names=classTree.classes_)

## Universal Bank Example

### Grow the full tree

In [ ]:
bank_df = pd.read_csv(my_drive_path + 'UniversalBank.csv')
bank_df = bank_df.drop(columns=['ID', 'ZIP Code'])

In [ ]:
bank_df

In [ ]:
X = bank_df.drop(columns=['Personal Loan'])
y = bank_df['Personal Loan']
train_X, valid_X, train_y, valid_y = train_test_split(X, y, test_size=0.4, random_state=1)

In [ ]:
fullClassTree = DecisionTreeClassifier()
fullClassTree.fit(train_X, train_y)

plotDecisionTree(fullClassTree, feature_names=train_X.columns)

## Confusion Matrix - Full Tree

In [ ]:
classificationSummary(train_y, fullClassTree.predict(train_X))
classificationSummary(valid_y, fullClassTree.predict(valid_X))

## Build a Smaller Tree -- Compare it will the Full Tree

In [ ]:
smallClassTree = DecisionTreeClassifier(max_depth=20, min_samples_split=5, min_impurity_decrease=0.0001)
smallClassTree.fit(train_X, train_y)

plotDecisionTree(smallClassTree, feature_names=X.columns, class_names=smallClassTree.classes_)

In [ ]:
classificationSummary(train_y, smallClassTree.predict(train_X))
classificationSummary(valid_y, smallClassTree.predict(valid_X))

## Cross Validation Example

In [ ]:
# Five-fold cross-validation of the full decision tree classifier
treeClassifier = DecisionTreeClassifier()

scores = cross_val_score(treeClassifier, train_X, train_y, cv=5)
print('Accuracy scores of each fold: ', [f'{acc:.3f}' for acc in scores])
print(f'Accuracy: {scores.mean():.3f} (+/- {scores.std() * 2:.3f})')
print(f'Accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})')

## Grid Search Example

In [ ]:
param_grid = {
    'max_depth': [5, 10, 15, 25],
    'min_samples_split': [2, 5, 10],
    'min_impurity_decrease': [0.00001, 0.0005, 0.01, 0.05]
}




gridSearch = GridSearchCV(DecisionTreeClassifier(), param_grid=param_grid, cv=5)
gridSearch.fit(train_X, train_y)

In [ ]:
print(gridSearch.best_score_)
print(gridSearch.best_params_)

In [ ]:
# well, because the initial check says max_depth 15, it does not mean that thats it.
# I have not checked say 13, 14 or any other close to 10 or 20

param_grid = {
    'max_depth': [12, 13, 14, 15, 16, 17, 18, 19],
    'min_samples_split': [4, 5, 6, 7],
    'min_impurity_decrease': [0.00001, 0.0005, 0.01, 0.05]
}




gridSearch = GridSearchCV(DecisionTreeClassifier(), param_grid=param_grid, cv=5)
gridSearch.fit(train_X, train_y)


In [ ]:
print(gridSearch.best_score_)
print(gridSearch.best_params_)

In [ ]:
bestClassTree = gridSearch.best_estimator_
bestClassTree

In [ ]:
plotDecisionTree(bestClassTree, feature_names=train_X.columns, class_names=bestClassTree.classes_)

In [ ]:
classificationSummary(train_y, bestClassTree.predict(train_X))
classificationSummary(valid_y, bestClassTree.predict(valid_X))

## Step By Step Example

In [ ]:
def basePlot(ax):
    mower_df.loc[mower_df.Ownership=='Owner'].plot(x='Income', y='Lot_Size', style='o',
                                                   markerfacecolor='C0', markeredgecolor='C0',
                                                   ax=ax)
    mower_df.loc[mower_df.Ownership=='Nonowner'].plot(x='Income', y='Lot_Size', style='o',
                                                      markerfacecolor='none', markeredgecolor='C1',
                                                      ax=ax)
    ax.legend(["Owner", "Nonowner"]);
    ax.set_xlim(20, 120)
    ax.set_ylim(13, 25)
    ax.set_xlabel('Income ($000s)')
    ax.set_ylabel('Lot Size (000s sqft)')
    return ax

fig, ax = plt.subplots(figsize=(7, 4))

ax = basePlot(ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax = basePlot(ax)
x0 = 59.7
ax.plot((x0, x0), (25, 13), color='grey')
plt.tight_layout()
plt.show()

In [ ]:
classTree = DecisionTreeClassifier(random_state=0, max_depth=1)
classTree.fit(mower_df.drop(columns=['Ownership']), mower_df['Ownership'])

plotDecisionTree(classTree, feature_names=mower_df.columns[:2], class_names=classTree.classes_)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax = basePlot(ax)
x0 = 59.7
y1 = 21.4
ax.plot((x0, x0), (25, 13), color='grey')
ax.plot((20, x0), (y1, y1), color='grey')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax = basePlot(ax)
x0 = 59.7
y1 = 21.4
y2 = 19.8
ax.plot((x0, x0), (25, 13), color='grey')
ax.plot((20, x0), (y1, y1), color='grey')
ax.plot((x0, 120), (y2, y2), color='grey')
plt.tight_layout()
plt.show()

In [ ]:
classTree = DecisionTreeClassifier(random_state=0, max_depth=2)
classTree.fit(mower_df.drop(columns=['Ownership']), mower_df['Ownership'])
plotDecisionTree(classTree, feature_names=mower_df.columns[:2], class_names=classTree.classes_)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax = basePlot(ax)
x0 = 59.7
y1 = 21.4
y2 = 19.8
x3 = 84.75
x4 = 61.5
ax.plot((x0, x0), (25, 13), color='grey')
ax.plot((20, x0), (y1, y1), color='grey')
ax.plot((x0, 120), (y2, y2), color='grey')
ax.plot((x3, x3), (13, y2), color='grey')
ax.plot((x4, x4), (13, y2), color='grey')
plt.tight_layout()
plt.show()

In [ ]:
classTree = DecisionTreeClassifier(random_state=0)
classTree.fit(mower_df.drop(columns=['Ownership']), mower_df['Ownership'])
plotDecisionTree(classTree, feature_names=mower_df.columns[:2], class_names=classTree.classes_)

## Extracting Info From the Tree

In [ ]:
tree = fullClassTree
print('Number of nodes', tree.tree_.node_count)

In [ ]:
estimator = tree
# Using those arrays, we can parse the tree structure:

n_nodes = estimator.tree_.node_count
children_left = estimator.tree_.children_left
children_right = estimator.tree_.children_right
feature = estimator.tree_.feature
threshold = estimator.tree_.threshold
value = estimator.tree_.value


# The tree structure can be traversed to compute various properties such
# as the depth of each node and whether or not it is a leaf.
node_depth = np.zeros(shape=n_nodes, dtype=np.int64)
is_leaves = np.zeros(shape=n_nodes, dtype=bool)
stack = [(0, -1)]  # seed is the root node id and its parent depth
while len(stack) > 0:
    node_id, parent_depth = stack.pop()
    node_depth[node_id] = parent_depth + 1

    # If we have a test node
    if (children_left[node_id] != children_right[node_id]):
        stack.append((children_left[node_id], parent_depth + 1))
        stack.append((children_right[node_id], parent_depth + 1))
    else:
        is_leaves[node_id] = True

from collections import Counter
nodeClassCounter = Counter()
terminal_leaves = 0
for i in range(n_nodes):
    if is_leaves[i]:
        terminal_leaves = terminal_leaves + 1
        nodeClassCounter.update([np.argmax(value[i][0])])
print()
print('Number of terminal leaves', terminal_leaves)
print(nodeClassCounter)

## Example

### Problem 9.2 Predicting Delayed Flights.

The file _FlightDelays.csv_ contains information on all commercial flights departing the Washington, DC area and arriving at New York during January 2004. For each flight, there is information on the departure and arrival airports, the distance of the route, the scheduled time and date of the flight, and so on. The variable that we are trying to predict is whether or not a flight is delayed. A delay is defined as an arrival that is at least 15 minutes later than scheduled.

__Data Preprocessing.__ Transform variable day of week (DAY_WEEK) info a categorical variable. Bin the scheduled departure time into eight bins. Use these and all other columns as predictors (excluding DAY_OF_MONTH). Partition the data into training (60%) and validation (40%) sets.

In [ ]:
# Load the data
delays_df = pd.read_csv(my_drive_path + 'FlightDelays.csv')
delays_df.head()

In [ ]:
# convert variable DAY_WEEK to categorical data type
delays_df['DAY_WEEK'].astype('category')

In [ ]:
# bin CRS_DEP_TIME variable into 8 bins
delays_df['binned_CRS_DEP_TIME'] = pd.cut(delays_df.CRS_DEP_TIME, 8, labels=False)
delays_df['binned_CRS_DEP_TIME'].astype('category')

In [ ]:
# remove DAY_OF_MONTH variable
predictors_df = delays_df
columns = list(delays_df.columns)
columns.remove('DAY_OF_MONTH')
predictors_df = predictors_df[columns]

### __9.2.a.__

__9.2.a.__ Fit a classification tree to the flight delay variable using all the relevant predictors. Do not include DEP_TIME (actual departure time) in the model because it is unknown at the time of prediction (unless we are generating our predictions of delays after the plane takes off, which is unlikely). Use a tree with maximum depth 8 and minimum impurity decrease = 0.01. Express the resulting tree as a set of rules.

In [ ]:
# select only those variables which can be used for predicting the outcome.
# create a new dataframe with predictors
columns = list(predictors_df.columns)
columns

columns.remove('CRS_DEP_TIME')
columns.remove('DEP_TIME')
columns.remove('FL_DATE')
columns.remove('FL_NUM')
columns.remove('TAIL_NUM')
columns.remove('Flight Status')
predictors_df = predictors_df[columns]
predictors_df.columns

predictors_df.head()

In [ ]:
# create dummies for categorical variables
predictors_df = pd.get_dummies(predictors_df, prefix_sep='_')
predictors_df.columns

In [ ]:
# partition the data into training (60%) and validation (40%) sets. set random_state=1 for the reproducibility of results
X = predictors_df
y = delays_df['Flight Status']

train_X, valid_X, train_y, valid_y = train_test_split(X, y, test_size=0.4, random_state=1)
train_X.head()

In [ ]:
# fit the tree model and draw tree
smallClassTree = DecisionTreeClassifier(max_depth=8, min_samples_split=50, min_impurity_decrease=0.01)
smallClassTree.fit(train_X, train_y)

print('Tree has {} nodes'.format(smallClassTree.tree_.node_count))
plotDecisionTree(smallClassTree, feature_names=train_X.columns)

The limited tree has only one splitting variable: Weather

If (Weather <= 0.5) then classify as Ontime.

### __9.2.b.__

__9.2.b.__ If you needed to fly between DCA and EWR on a Monday at 7:00 AM, would you be able to use this tree? What other information would you need? Is it available in practice? What information is redundant?

__Answer:__

We cannot use this tree, because we must know the Weather. The redundant information is the day of week (Monday) and arrival airport (EWR). The tree requires knowing whether the weather was inclement or not. We may not know the weather in advance.

### __9.2.c.__

__9.2.c.__ Fit the same tree as in (a), this time excluding the Weather predictor. Display both the resulting (small) tree and the full-grown tree. You will find that the small tree contains a single terminal node.

In [ ]:
# remove variable Weather from the analysis
predictors1_df = predictors_df
columns = list(predictors_df.columns)
columns
columns.remove('Weather')
predictors1_df = predictors1_df[columns]
predictors1_df.columns

In [ ]:
X1 = predictors1_df
y1 = delays_df['Flight Status']

train_X1, valid_X1, train_y1, valid_y1 = train_test_split(X1, y1, test_size=0.4, random_state=1)

# full-grown tree
fullClassTree = DecisionTreeClassifier()
fullClassTree.fit(train_X1, train_y1)

print('Tree has {} nodes'.format(fullClassTree.tree_.node_count))
plotDecisionTree(fullClassTree, feature_names=train_X1.columns)

In [ ]:
# small tree
ClassTree = DecisionTreeClassifier(max_depth=8, min_samples_split=50, min_impurity_decrease=0.01)
ClassTree.fit(train_X1, train_y1)

print('Tree has {} nodes'.format(ClassTree.tree_.node_count))
plotDecisionTree(ClassTree, feature_names=train_X1.columns)

__9.2.c.i.__ How is the small tree used for classification? (What is the rule for classifying?)

__Answer:__

In the small tree we get a single terminal node labeled "ontime." Therefore any new flight will be classified as being "on time".

__9.2.c.ii__ To what is this rule equivalent?

__Answer:__

This is equivalent to the naïve rule, which is the majority rule. In this dataset most of the flights arrived on time, and therefore the naïve rule is to classify a new flight as arriving on time.

__9.2.c.iii.__ Examine the full-grown tree. What are the top three predictors according to this tree?

CARRIER=US, CARRIER=DL, binned_CRS_DEP_TIME.

__9.2.c.iv.__ Why, technically, does the small tree result in a single node?

__Answer:__

The small tree results in a single node because adding splits would violate one of the constraints we used to limit tree growth.

__9.2.c.v.__ What is the disadvantage of using the top levels of the full-grown tree as opposed to the small tree?

__Answer:__

Simply using the top layers of the full decision tree would be an ad hoc visual approach, and does not assure an optimal solution. Using "gridsearchCV" in Python allows us to set the parameters for limiting tree growth by assessing error on the validation data. We did not use "gridsearchCV" in this case, opting to keep the problem simple by specifying the limiting parameters.

__9.2.c.vi.__ Compare this general result to that from logistic regression in the example in Chapter 10. What are possible reasons for the classification tree’s failure to find a good predictive model?

In [ ]:
# predictive power of tree
# predicted values for validation set
pred_v = ClassTree.predict(valid_X1)
# confusion matrix for validation set
classificationSummary(valid_y1, pred_v)

The logistic regression improves only marginally on the naive rule, and the simple tree we ended up with improves on it not at all, so it is likely that the predictor variables offer little predictive power. With poor predictive power and a relatively small dataset, the model-based logistic regression may do a bit better by virtue of imposing structure, as opposed to the tree, which is more at the mercy of the data and can suffer from instability.

## Overfitting Plot

Interpreting the Overfitting Curve:

    - We want to understand how different hyperparameters affect performance
    - Typically we will have the plot of accuracy and a hyperparameter value
    
Training Accuracy (Training Curve):

    - The training curve shows how well the model fits the training data as the hyperparameter value changes.
    - At the beginning, the training accuracy increases as the model's complexity (e.g., max_depth) increases. This is because  the model is capturing more details and patterns in the training data.
    - As the hyperparameter value continues to increase, the training accuracy approaches or reaches 100%. This is an indicator of potential overfitting because the model has become too complex and starts fitting noise in the data.

Validation Accuracy (Validation Curve):

    - The validation curve represents the model's performance on a separate validation dataset, which was not used during training.
    - Initially, the validation accuracy tends to increase as the hyperparameter value grows, as the model gets better at generalizing from the training data.
    - There comes a point where the validation accuracy plateaus or begins to decrease, even if the training accuracy continues to improve. This is a sign of overfitting.
    - The hyperparameter value at which the validation accuracy is the highest (the peak of the curve) often corresponds to the optimal model complexity.

Interpreting the curve may lead to the following insights:

 - Underfitting: If both training and validation accuracy are low and the curves are flat, the model is too simple and underfits the data. You may need to increase the model's complexity (e.g., increase max_depth or add more features).

- Optimal Model Complexity: The hyperparameter value at which the validation accuracy peaks represents the optimal model complexity, striking a balance between bias and variance. This is the value to choose for your final model.

- Overfitting: If the training accuracy continues to increase while the validation accuracy decreases or remains flat, the model is overfitting. The optimal model complexity is before this point, and you should select a hyperparameter value corresponding to the peak of the validation curve.

In summary, the overfitting curve helps you identify the optimal hyperparameter value that leads to the best model performance on unseen data, while avoiding overfitting. It guides the hyperparameter tuning process to build a model with good generalization capabilities.

In [ ]:
#Example of an overfitting curve


# Define a range of tree depths to explore
depth_range = range(1, 21)

# Initialize empty lists to store training and validation accuracy
train_accuracy = []
valid_accuracy = []

# Iterate through different tree depths and build and evaluate models
for depth in depth_range:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=0)
    clf.fit(train_X1, train_y1)

    train_accuracy.append(clf.score(train_X1, train_y1))
    valid_accuracy.append(clf.score(valid_X1, valid_y1))

# Plot the overfitting curve
plt.figure(figsize=(10, 6))
plt.plot(depth_range, train_accuracy, label="Training Accuracy", marker='o')
plt.plot(depth_range, valid_accuracy, label="Validation Accuracy", marker='o')
plt.xlabel("Tree Depth")
plt.ylabel("Accuracy")
plt.title("Overfitting Curve for DecisionTreeClassifier")
plt.legend()
plt.grid(True)
plt.show()